# ES vs Noise Comparison

Compare Vicuna ES results against Noise attack results using only the common query IDs.

## Summary Statistics

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style='whitegrid')
ROOT = Path('..').resolve()

def latest_results(base: Path, pattern: str) -> Path:
    runs = sorted(p for p in base.glob(pattern) if p.is_dir())
    if not runs:
        raise FileNotFoundError(f'No runs found under {base / pattern}')
    return runs[-1] / 'final_results' / 'results.csv'

def load_results(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df['query_id'] = df['query_id'].astype(str)
    return df

es_path = latest_results(ROOT / 'outputs' / 'vicuna_7b', 'run_*')
noise_path = latest_results(ROOT / 'outputs' / 'noise_attack', 'run_*')
es = load_results(es_path)
noise = load_results(noise_path)

common = set(es['query_id']) & set(noise['query_id'])
print('Common queries:', len(common))
es = es[es['query_id'].isin(common)].copy()
noise = noise[noise['query_id'].isin(common)].copy()

es_success = es['attack_success'].astype(str).eq('True').mean()
noise_success = noise['attack_success'].astype(str).eq('True').mean()
print('ES success:', es_success)
print('Noise success:', noise_success)
print('ES fitness mean:', pd.to_numeric(es['fitness']).mean())
print('Noise fitness mean:', pd.to_numeric(noise['fitness']).mean())

## Visualization

In [ ]:
summary = pd.DataFrame({
    'method': ['ES', 'Noise'],
    'success_rate': [es_success, noise_success],
    'avg_fitness': [pd.to_numeric(es['fitness']).mean(), pd.to_numeric(noise['fitness']).mean()],
    'success_count': [es['attack_success'].astype(str).eq('True').sum(), noise['attack_success'].astype(str).eq('True').sum()],
})
display(summary)

plt.figure(figsize=(6, 4))
sns.barplot(data=summary, x='method', y='success_rate', palette='deep')
plt.title('Attack Success Rate')
plt.xlabel('Method')
plt.ylabel('Success Rate')
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.histplot(pd.to_numeric(es['fitness']), label='ES', kde=True, stat='density', bins=20, color='steelblue', alpha=0.5)
sns.histplot(pd.to_numeric(noise['fitness']), label='Noise', kde=True, stat='density', bins=20, color='darkorange', alpha=0.5)
plt.title('Fitness Distribution')
plt.xlabel('Fitness')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=[pd.to_numeric(es['fitness']), pd.to_numeric(noise['fitness'])])
plt.xticks([0, 1], ['ES', 'Noise'])
plt.title('Fitness Comparison')
plt.xlabel('Method')
plt.ylabel('Fitness')
plt.tight_layout()
plt.show()

## Insights

- Does readability matter? Compare the success and fitness profiles to see whether the more semantic ES objective gains measurable advantage.
- Does noise help or hurt? Inspect whether noise achieves comparable success counts or simply raises variance without stable gains.